# 🚀 NV Super Soldier (NVSS) — Google Colab Pro ワンクリック GPU 強化学習

このノートブックは **Google Colab Pro (A100 / L4 / V100 / T4 GPU)** 上で **数千〜12,288試合を同時並列シミュレーション** し、
**NV Super Soldier (NVSS)** を強化学習（PPO + 未来機能ゼロショット適応 + 10倍人間味報酬設計）で鍛え上げるためのワンクリック学習パイプラインです。

### 使い方（ワンクリック）
1. メニュー **「ランタイム」→「ランタイムのタイプを変更」** で **GPU（A100 または L4 推奨）** を選択。
2. 下の **「学習設定」** で目標ステップ数を選び、**「ランタイム」→「すべてのセルを実行 (`Ctrl+F9`)」** を押すだけ！
3. Google Drive にチェックポイントを自動保存・自動再開するため、複数セッションに渡って **数億〜数十億ステップ／試合規模** まで無限に強化できます。
4. 学習完了後に自動ダウンロードされる **`nvss-weights.json`** を、ゲームの対戦設定画面 **「ウェイト読込 (.json)」** ボタンに読み込ませるだけで即座に最強NVSSが反映されます！

In [ ]:
#@title ⚙️ ステップ1: 環境セットアップ・GPU自動検出・Drive自動保存マウント { display-mode: "form" }
REPO_URL = "https://github.com/sc8z35a-collab/Result.git" #@param {type:"string"}
USE_GOOGLE_DRIVE = True #@param {type:"boolean"}
DRIVE_SAVE_DIR = "/content/drive/MyDrive/NVSS_Colab_Pro_Runs" #@param {type:"string"}

import os, subprocess, torch

if torch.cuda.is_available():
    gpu = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"✅ GPU 検出: {gpu} ({vram:.1f} GB VRAM) | bf16={torch.cuda.is_bf16_supported()}")
else:
    print("⚠️ GPU が無効です。ランタイム → ランタイムのタイプを変更 → GPU (A100/L4) を選択してください！")

if USE_GOOGLE_DRIVE:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        OUT_DIR = DRIVE_SAVE_DIR
    except Exception as e:
        print(f"Drive マウントをスキップ ({e}) -> ローカル保存を使用")
        OUT_DIR = "/content/nvss_runs"
else:
    OUT_DIR = "/content/nvss_runs"

os.makedirs(OUT_DIR, exist_ok=True)

if not os.path.exists("/content/Result"):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "/content/Result"], check=True)
else:
    subprocess.run(["git", "-C", "/content/Result", "pull", "--ff-only"], check=False)

print(f"✅ セットアップ完了！チェックポイント保存先: {OUT_DIR}")

In [ ]:
#@title 🔥 ステップ2: NV Super Soldier (NVSS) ワンクリック GPU 強化学習スタート { display-mode: "form" }
PRESET = "Pro: 1億ステップ (A100で約10〜20分)" #@param ["Quick: 2,000万ステップ (約2〜4分)", "Pro: 1億ステップ (A100で約10〜20分)", "Ultra: 10億ステップ (Drive自動再開で継続強化)", "Billion+: 50億ステップ (超長期強化)"]
RESET_CHECKPOINT = False #@param {type:"boolean"}

preset_map = {
    "Quick: 2,000万ステップ (約2〜4分)": 20_000_000,
    "Pro: 1億ステップ (A100で約10〜20分)": 100_000_000,
    "Ultra: 10億ステップ (Drive自動再開で継続強化)": 1_000_000_000,
    "Billion+: 50億ステップ (超長期強化)": 5_000_000_000,
}
TARGET_STEPS = preset_map[PRESET]

cmd = [
    "python3", "/content/Result/colab/train_nvss.py",
    "--target-steps", str(TARGET_STEPS),
    "--out-dir", OUT_DIR,
    "--log-every", "5",
    "--save-every", "20",
]
if RESET_CHECKPOINT:
    cmd.append("--fresh")

print("実行コマンド:", " ".join(cmd))
import subprocess, sys
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    sys.stdout.write(line)
    sys.stdout.flush()
rc = proc.wait()
assert rc == 0, f"Training exited with code {rc}"

In [ ]:
#@title 📈 ステップ3: 学習曲線（勝率・10×人間味スコア・報酬）を可視化
import json, os
import matplotlib.pyplot as plt

metrics_file = os.path.join(OUT_DIR, "training_metrics.json")
with open(metrics_file, "r", encoding="utf-8") as f:
    hist = json.load(f)

steps_m = [h["steps"] / 1e6 for h in hist]
wins = [h["win_pct"] for h in hist]
human = [h["human_realism"] for h in hist]
rews = [h["mean_reward"] for h in hist]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(steps_m, wins, color="#00d4aa", lw=2); axes[0].set_title("Win Rate (%)"); axes[0].set_xlabel("Steps (M)"); axes[0].grid(alpha=0.3)
axes[1].plot(steps_m, human, color="#4aa8ff", lw=2); axes[1].set_title("Human Realism Index (/100)"); axes[1].set_xlabel("Steps (M)"); axes[1].grid(alpha=0.3)
axes[2].plot(steps_m, rews, color="#ff8844", lw=2); axes[2].set_title("Mean Tactical Reward"); axes[2].set_xlabel("Steps (M)"); axes[2].grid(alpha=0.3)
plt.tight_layout()
plt.show()
print(f"最新成績: {hist[-1]['steps']:,} steps | {hist[-1]['matches']:,} 試合 | 勝率 {hist[-1]['win_pct']:.1f}% | 人間味 {hist[-1]['human_realism']:.1f}/100")

In [ ]:
#@title 💾 ステップ4: 学習済み NVSS ウェイト (nvss-weights.json) をダウンロード
import os
weights_json = os.path.join(OUT_DIR, "nvss-weights.json")
assert os.path.exists(weights_json), "nvss-weights.json が見つかりません"
size_kb = os.path.getsize(weights_json) / 1024
print(f"✅ 出力ファイル準備完了: {weights_json} ({size_kb:.1f} KB)")
print("👉 Result のロビー画面にある「ウェイト読込 (.json)」ボタンで読み込むと、この最強NVSSが即座に反映されます！")
try:
    from google.colab import files
    files.download(weights_json)
except Exception:
    print("ローカル実行環境のためブラウザ自動ダウンロードをスキップしました。")